# ASHP Techno-Economic Assessment — LiverpoolA walkthrough of the model behind the results in the README. Each sectionmirrors one module in `src/`.The question: under 2025 UK energy prices, does replacing a gas boiler with anair source heat pump make financial sense for a Liverpool household?

In [ ]:
import syssys.path.append('..')import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom src import config, weather, heatpump, demand, economics, sensitivity, plottingpd.set_option('display.max_columns', None)pd.set_option('display.width', 120)

## 1 — WeatherFive years of hourly observations for Liverpool, collapsed into onerepresentative year. Averaging by (month, day, hour) keeps the diurnal andseasonal shape the COP model needs while damping any single unusual winter.The design temperature is the 1st percentile of the whole hourly record. Thatis the same quantity MCS MIS 3005-D calls the 99th percentile exceedancetemperature — the value outdoor air stays above for 99% of the year.

In [ ]:
profile, t_design = weather.load_weather()stats = weather.summarise(profile, t_design)for k, v in stats.items():    print(f'{k:22} {v}')

In [ ]:
monthly = profile.groupby('month')['outdoor_temp_C'].agg(['mean','min','max']).round(1)monthly.index = plotting.MONTHSmonthly

Note how narrow that range is. A maritime climate means the heat pump rarelysees its design condition — which is why, as we'll see, seasonal COP comes outwell above the rated figure.

## 2 — Flow temperatureA weather-compensating controller delivers only as much flow temperature as theoutdoor condition demands. Two anchors define the curve: 55 °C at the designoutdoor temperature, 35 °C at the balance point above which internal and solargains cover the load.The linear shape isn't arbitrary. Building heat loss is proportional to theindoor–outdoor difference (BS EN 12831-1), and radiator output is roughly linearin mean water-to-air difference over this range (CIBSE Guide A). Combine the twoand flow temperature falls out linear in outdoor temperature.

In [ ]:
x = np.linspace(-5, 20, 200)y = heatpump.flow_temperature(x, t_design)for t in [-2, t_design, 0, 7, 10, config.T_BALANCE_POINT_C, 18]:    print(f'{t:6.1f} °C outdoor  →  {heatpump.flow_temperature(np.array([t]), t_design)[0]:5.1f} °C flow')

The 7 °C row matters. That's the *A7* of the A7/W55 test condition used forcalibration — and the curve puts flow at about 45 °C there, not 55 °C. A7/W55 isa laboratory reference, not an operating state. Flow only reaches 55 °C at thedesign condition.

In [ ]:
plotting.plot_weather_compensation(t_design)

## 3 — Calibrating the COP modelStaffell et al. (2012) established that real heat pumps achieve a roughlyconstant fraction η of the Carnot maximum across their operating range. Thatmakes the model tractable: anchor η to one measured datapoint, then projectacross every hour.For space heating the anchor is the manufacturer's COP at A7/W55. For hot waterthere is no equivalent test point, so η is instead set so the simulated annualmean reproduces the MCS 031 a) seasonal performance factor of 1.70.

In [ ]:
print(f'Carnot COP at A7/W55: {config.COP_CARNOT_REF:.3f}')print(f'  = {config.T_FLOW_REF_K} / ({config.T_FLOW_REF_K} - {config.T_AIR_REF_K})\n')heatpump.efficiency_factors(profile)

η around 0.42 means the machine achieves 42% of the theoretical maximum —squarely in the 0.30–0.45 band Staffell et al. report for modern residentialunits. The Flat sits lower because its smaller unit carries a lower rated COP.

## 4 — Hourly COPApplying η across the year. Space heating COP swings with both source and sink;hot water only with source, since the cylinder target never moves.

In [ ]:
dwelling = 'Semi-Detached'cop_df = heatpump.build_cop_profile(profile, t_design, dwelling)monthly_cop = cop_df.groupby('month')[['outdoor_temp_C','flow_temp_C','cop_space','cop_dhw']].mean().round(2)monthly_cop.index = plotting.MONTHSmonthly_cop

Read across January: coldest outdoor, highest flow temperature, lowest COP.Then July: mildest outdoor, minimum flow, highest COP. The inverse relationshipbetween flow temperature and efficiency is the whole mechanism.Hot water COP varies far less — it has no weather compensation to help it, onlythe seasonal swing in source temperature.

In [ ]:
plotting.plot_seasonal_profile(cop_df, dwelling)

## 5 — Demand and simulationAnnual demands come from DECC benchmarks, split 88/12 between space heating andhot water. Space heating is spread across the year by degree-hour weighting;hot water evenly, since people shower in August too.

In [ ]:
demand.demand_table()

In [ ]:
results = []profiles = {}for d in config.DWELLING_TYPES:    df = heatpump.build_cop_profile(profile, t_design, d)    df = demand.distribute(df, d)    df = economics.hourly_electricity(df)    profiles[d] = df    results.append(economics.appraise(df, d))results = pd.DataFrame(results)results[['Dwelling type','SCOP space','SCOP DHW','SCOP total']]

### Does the model hold up?Two checks. The hot water figure should reproduce the MCS target it wascalibrated to — it does, to within about 1%. The small excess comes fromJensen's inequality: the Carnot relationship is convex, so averaging hourly COPsgives slightly more than the COP at the average temperature.Space heating is checked against the Electrification of Heat field data — 742monitored UK installations, the best evidence available.

In [ ]:
scop = results['SCOP total'].mean()print(f'Simulated SCOP total   {scop:.2f}')print(f'EoH field SPFH2        {config.EOH_SPFH2_MEDIAN:.2f}   gap +{scop - config.EOH_SPFH2_MEDIAN:.2f}')print(f'EoH field SPFH4        {config.EOH_SPFH4_MEDIAN:.2f}   gap +{scop - config.EOH_SPFH4_MEDIAN:.2f}')

The model runs high, and the reason is known: the Carnot framework has nodefrost cycles, no part-load cycling penalty, no circulation pumps. SPFH4includes all of it.Worth being clear about which way this bias cuts. Real performance being *worse*than modelled means real running costs are *higher* than reported here — so theeconomic case against switching is, if anything, stronger than what follows.

## 6 — The economicsCost per unit of delivered heat is the cleanest way to see it. A gas boilerwastes 10% of what it burns; a heat pump multiplies what it draws by its SCOP.

In [ ]:
hc = economics.heat_cost_comparison(scop)print(f"Gas boiler   {config.GAS_PRICE_P_PER_KWH:.2f}p ÷ {config.BOILER_EFFICIENCY:.2f}  = {hc['gas_p_per_kWh_heat']:.2f} p/kWh heat")print(f"ASHP         {config.OFGEM_ANNUAL_AVG_P:.2f}p ÷ {scop:.2f}  = {hc['ashp_p_per_kWh_heat']:.2f} p/kWh heat")print(f"\nHeat pump premium: {hc['gap_p_per_kWh']:.2f} p/kWh ({hc['ashp_premium_pct']:.1f}%)")

A SCOP above 3 and it still loses. The efficiency advantage is real but theprice ratio is bigger.

In [ ]:
results[['Dwelling type','Gas cost (GBP/yr)','ASHP cost (GBP/yr)',         'Annual saving (GBP)','NPV 15yr (GBP)','Payback (yr)']]

Payback is `NaN` throughout, and that is not a bug. Payback assumes savingsaccumulate toward the capital cost. Here savings are negative, so the gap widensevery year — there is no year in which it closes.Same reason the 20-year NPV is worse than the 10-year.

In [ ]:
horizons = [c for c in results.columns if c.startswith('NPV')]results.set_index('Dwelling type')[horizons]

In [ ]:
plotting.plot_running_costs(results)plotting.plot_npv(results)

## 7 — What price would work?Set the two heat costs equal and solve for the electricity price.

In [ ]:
results[['Dwelling type','SCOP total','Break-even price (p/kWh)',         'Required reduction (p/kWh)','Required reduction (%)']]

In [ ]:
q = config.OFGEM_QUARTERLY_Pbe = results['Break-even price (p/kWh)'].min()print('Ofgem 2025 quarterly rates vs break-even:\n')for quarter, rate in q.items():    print(f'  Q{quarter}   {rate:.2f}p   {rate - be:+.2f}p above the lowest break-even')

Every quarter, every dwelling type. Even Q1 — the cheapest quarter of theyear — sits above the threshold.This answers one of the questions the project set out to test: whether quarterlytariff structure changes the picture relative to using an annual average. Itmoves the numbers, but not the conclusion.

In [ ]:
plotting.plot_breakeven(results)

## 8 — What would change the answerSwing each input ±20% and re-solve. Note the swing goes on the *input*, not onthe NPV — the response isn't proportional, because price variables act throughthe annual saving while capital cost acts on the capital term alone.

In [ ]:
tornado_df = sensitivity.tornado_all(results)(tornado_df.groupby('Variable')['Swing width (GBP)']           .mean().sort_values(ascending=False)           .to_frame('Mean NPV swing (GBP)').round(0))

Electricity price dominates. Installed cost — the thing current policytargets through capital grants — is the weakest lever of the three.

In [ ]:
plotting.plot_tornado(tornado_df)

In [ ]:
grants = sensitivity.grant_table(results)grants

The required grant exceeds the installed cost, which looks odd until youremember what it has to do: cover the capital *and* offset fifteen years ofdiscounted running cost penalty.Notice the shortfall grows with dwelling size. A flat-rate grant systematicallyunder-supports larger homes — which are also the homes where switching wouldabate the most carbon.

In [ ]:
plotting.plot_grant_gap(grants)

## 9 — Policy scenarios

In [ ]:
scenarios = sensitivity.policy_scenarios(results)scenarios.pivot(index='Dwelling type', columns='Scenario', values='NPV 15yr (GBP)')

Levy rebalancing — moving environmental and social levies off electricitybills, worth roughly 6 p/kWh per Rosenow et al. (2025) — flips every dwellingtype positive on its own.Grant uplift reaches nominal payback but leaves the operational economicsuntouched: the household still pays more every year to run the heat pump. Thatdistinction matters for whether a subsidy is politically durable.

## ConclusionSeasonal efficiency of 3.2–3.3 against an electricity-to-gas price ratio near4:1. The efficiency is real and the technology works — the arithmetic justdoesn't close at current prices.Break-even needs electricity at 23.27–23.81 p/kWh. Every Ofgem 2025 quarterlyrate sits above that. Fabric improvement shrinks the absolute penalty but cannotchange its sign; only the price ratio can.And since the Carnot model runs optimistic against field data, the real positionis slightly worse than these numbers show.